# Data Analysis — 20 Preguntas de Negocio

Todas las consultas sobre `ANALYTICS.OBT_TRIPS` usando Spark.

## Imports y configuración

In [3]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_ANALYTICS    = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

spark = SparkSession.builder \
    .appName("P3_DataAnalysis") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.driver.extraJavaOptions", "-XX:MaxDirectMemorySize=4g") \
    .config("spark.executor.extraJavaOptions", "-XX:MaxDirectMemorySize=4g") \
    .getOrCreate()

sf_options = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_ANALYTICS,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
    "use_arrow"  : "false",
    "sfCompress" : "on",
}

def query_obt(sql):
    """Helper para hacer queries a OBT_TRIPS sin cargar toda la tabla"""
    return spark.read \
        .format("net.snowflake.spark.snowflake") \
        .options(**sf_options) \
        .option("query", sql) \
        .load()

df = (
     spark.read.format("net.snowflake.spark.snowflake")
     .options(**sf_options)
     .option("dbtable", "OBT_TRIPS")
     .load()
)

df = df.toDF(*[c.lower() for c in df.columns])
print(df.columns)

['pickup_datetime', 'dropoff_datetime', 'pu_location_id', 'pu_zone', 'pu_borough', 'do_location_id', 'do_zone', 'do_borough', 'service_type', 'vendor_id', 'vendor_name', 'rate_code_id', 'rate_code_desc', 'payment_type', 'payment_type_desc', 'trip_type', 'passenger_count', 'trip_distance', 'store_and_fwd_flag', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount', 'run_id', 'ingested_at_utc', 'source_path', 'source_year', 'source_month', 'pickup_date', 'pickup_hour', 'dropoff_date', 'dropoff_hour', 'day_of_week', 'month', 'year', 'trip_duration_min', 'avg_speed_mph', 'tip_pct']


## a) Top 10 zonas de pickup por volumen mensual

In [ ]:
result = (df.groupby("pu_zone", "source_month")
  .agg(F.count("*").alias("num_trips"))
  .orderBy(F.col("num_trips").desc())
  .limit(10)
)
result.show(truncate=False)

In [8]:
raw = query_obt("""
    SELECT pu_zone AS pickup_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY pu_zone, source_year, source_month
""")

(raw.orderBy(F.col("num_trips").desc())
   .limit(10)
   .show(truncate=False))

+---------------------+-----------+------------+---------+
|PICKUP_ZONE          |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Upper East Side South|2015       |4           |497454   |
|Upper East Side South|2015       |5           |484375   |
|Upper East Side South|2015       |3           |473179   |
|Upper East Side South|2015       |10          |468200   |
|Upper East Side South|2015       |1           |466871   |
|Upper East Side South|2016       |5           |464972   |
|Midtown Center       |2015       |3           |463829   |
|Midtown Center       |2015       |4           |460601   |
|Upper East Side North|2015       |4           |457852   |
|Midtown Center       |2016       |3           |457522   |
+---------------------+-----------+------------+---------+



In [9]:
query_obt("""
    SELECT pu_zone AS pickup_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY pu_zone, source_year, source_month
    ORDER BY num_trips DESC
    LIMIT 10
""").show(truncate=False)

+---------------------+-----------+------------+---------+
|PICKUP_ZONE          |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Upper East Side South|2015       |4           |497454   |
|Upper East Side South|2015       |5           |484375   |
|Upper East Side South|2015       |3           |473179   |
|Upper East Side South|2015       |10          |468200   |
|Upper East Side South|2015       |1           |466871   |
|Upper East Side South|2016       |5           |464972   |
|Midtown Center       |2015       |3           |463829   |
|Midtown Center       |2015       |4           |460601   |
|Upper East Side North|2015       |4           |457852   |
|Midtown Center       |2016       |3           |457522   |
+---------------------+-----------+------------+---------+



## b) Top 10 zonas de dropoff por volumen mensual

In [ ]:
result = (df.groupby("do_zone", "source_year", "source_month")
  .agg(F.count("*").alias("num_trips"))
  .orderBy(F.col("num_trips").desc())
  .limit(10)
)

result.show(truncate=False)

In [11]:
raw = query_obt("""
    SELECT do_zone AS dropoff_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY do_zone, source_year, source_month
""")

(raw.orderBy(F.col("num_trips").desc())
  .limit(10)
  .show(truncate=False))

+---------------------+-----------+------------+---------+
|DROPOFF_ZONE         |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Midtown Center       |2015       |3           |506655   |
|Midtown Center       |2015       |4           |502105   |
|Midtown Center       |2015       |5           |480298   |
|Midtown Center       |2015       |6           |474849   |
|Midtown Center       |2015       |1           |472752   |
|Upper East Side North|2015       |4           |468453   |
|Midtown Center       |2016       |3           |467509   |
|Upper East Side North|2015       |5           |464553   |
|Midtown Center       |2015       |7           |463701   |
|Midtown Center       |2015       |2           |461932   |
+---------------------+-----------+------------+---------+



In [12]:
query_obt("""
    SELECT do_zone AS dropoff_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY do_zone, source_year, source_month
    ORDER BY num_trips DESC
    LIMIT 10
""").show(truncate=False)

+---------------------+-----------+------------+---------+
|DROPOFF_ZONE         |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Midtown Center       |2015       |3           |506655   |
|Midtown Center       |2015       |4           |502105   |
|Midtown Center       |2015       |5           |480298   |
|Midtown Center       |2015       |6           |474849   |
|Midtown Center       |2015       |1           |472752   |
|Upper East Side North|2015       |4           |468453   |
|Midtown Center       |2016       |3           |467509   |
|Upper East Side North|2015       |5           |464553   |
|Midtown Center       |2015       |7           |463701   |
|Midtown Center       |2015       |2           |461932   |
+---------------------+-----------+------------+---------+



## c) Evolucion mensual de total_amount y tip_pct por borough

In [ ]:
result = (df.groupby("pu_borough", "source_year", "source_month")
  .agg(
       F.avg("total_amount").alias("avg_total_amount")
       F.avg("tip_pct").alias("tip_pct")
      )
  .orderBy("source_year", "source_month")
)  

result.show(truncate=False)

In [13]:
raw = query_obt("""
    SELECT pu_borough as pickup_borough, source_year, source_month, AVG(total_amount), AVG(tip_pct)
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month
""")

(raw.orderBy("source_year", "source_month")
    .show(truncate=False)
)  

+--------------+-----------+------------+-------------------+------------------+
|PICKUP_BOROUGH|SOURCE_YEAR|SOURCE_MONTH|"AVG(TOTAL_AMOUNT)"|"AVG(TIP_PCT)"    |
+--------------+-----------+------------+-------------------+------------------+
|Queens        |2015       |1           |29.496841686927695 |7.329751541341815 |
|Manhattan     |2015       |1           |13.753432942842919 |9.26580947765827  |
|EWR           |2015       |1           |78.58980645161289  |10.984210955859616|
|Unknown       |2015       |1           |15.85125789326167  |9.481767524998867 |
|N/A           |2015       |1           |53.491176470588236 |8.793154205268566 |
|Staten Island |2015       |1           |32.77122194513716  |5.888912290299136 |
|Brooklyn      |2015       |1           |15.732932523393528 |8.9229088793002   |
|Bronx         |2015       |1           |12.93515529247911  |2.1896850648584416|
|Brooklyn      |2015       |2           |16.22991758934122  |9.174408114251955 |
|EWR           |2015       |

In [14]:
query_obt("""
    SELECT pu_borough as pickup_borough, source_year, source_month, AVG(total_amount), AVG(tip_pct)
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month
    ORDER BY source_year, source_month
""").show(truncate=False)

+--------------+-----------+------------+-------------------+------------------+
|PICKUP_BOROUGH|SOURCE_YEAR|SOURCE_MONTH|"AVG(TOTAL_AMOUNT)"|"AVG(TIP_PCT)"    |
+--------------+-----------+------------+-------------------+------------------+
|Brooklyn      |2015       |1           |15.732932523393528 |8.922908879300202 |
|N/A           |2015       |1           |53.49117647058823  |8.793154205268566 |
|Manhattan     |2015       |1           |13.753432942842919 |9.26580947765827  |
|Queens        |2015       |1           |29.49684168692769  |7.329751541341814 |
|EWR           |2015       |1           |78.5898064516129   |10.984210955859618|
|Unknown       |2015       |1           |15.851257893261668 |9.481767524998867 |
|Staten Island |2015       |1           |32.771221945137164 |5.888912290299135 |
|Bronx         |2015       |1           |12.93515529247911  |2.189685064858441 |
|Unknown       |2015       |2           |16.242039714742404 |9.857849160031572 |
|Brooklyn      |2015       |

## d) Ticket promedio por service_type y mes

In [ ]:
result = (
   df.groupby("service_type", "source_year", "source_month")
     .agg(F.avg(total_amount).alias("avg_total_amount"))
     .orderBy("source_year", "source_month", "service_type")
)

result.show(truncate=False)

In [16]:
raw = query_obt("""
    SELECT service_type, source_year, source_month,
           AVG(total_amount) AS avg_total_amount
    FROM OBT_TRIPS
    GROUP BY service_type, source_year, source_month
""")

(raw.orderBy("source_year", "source_month", "service_type")
    .show(truncate=False)
)

+------------+-----------+------------+------------------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|AVG_TOTAL_AMOUNT  |
+------------+-----------+------------+------------------+
|green       |2015       |1           |14.820434363049914|
|yellow      |2015       |1           |15.108694711828159|
|green       |2015       |2           |14.524728774508837|
|yellow      |2015       |2           |15.40038155373236 |
|green       |2015       |3           |14.61037330989992 |
|yellow      |2015       |3           |15.847585113439196|
|green       |2015       |4           |14.8497854274392  |
|yellow      |2015       |4           |16.035191014569467|
|green       |2015       |5           |15.281361374811409|
|yellow      |2015       |5           |16.421437168258205|
|green       |2015       |6           |14.990150579316087|
|yellow      |2015       |6           |16.37907894914255 |
|green       |2015       |7           |14.945052789404231|
|yellow      |2015       |7           |16.13061124758746

In [17]:
query_obt("""
    SELECT service_type, source_year, source_month,
           AVG(total_amount) AS avg_total_amount
    FROM OBT_TRIPS
    GROUP BY service_type, source_year, source_month
    ORDER BY source_year, source_month, service_type
""").show(truncate=False)

+------------+-----------+------------+------------------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|AVG_TOTAL_AMOUNT  |
+------------+-----------+------------+------------------+
|green       |2015       |1           |14.820434363049914|
|yellow      |2015       |1           |15.108694711828157|
|green       |2015       |2           |14.524728774508837|
|yellow      |2015       |2           |15.400381553732357|
|green       |2015       |3           |14.610373309899916|
|yellow      |2015       |3           |15.847585113439193|
|green       |2015       |4           |14.8497854274392  |
|yellow      |2015       |4           |16.035191014569463|
|green       |2015       |5           |15.281361374811407|
|yellow      |2015       |5           |16.421437168258205|
|green       |2015       |6           |14.990150579316088|
|yellow      |2015       |6           |16.37907894914255 |
|green       |2015       |7           |14.945052789404228|
|yellow      |2015       |7           |16.13061124758746

## e) Viajes por hora del dia y dia de semana (picos)

In [ ]:
result = (
    df.groupby("day_of_week", "pickup_hour")
      .agg(F.count("*").alias("num_trips"))
      .orderBy("day_of_week", "pickup_hour")
)

result.show(truncate=False)

In [18]:
raw = query_obt("""
    SELECT day_of_week, pickup_hour, COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY day_of_week, pickup_hour
""")

(raw.orderBy("day_of_week", "pickup_hour")
      .show(truncate=False)
)

+-----------+-----------+---------+
|DAY_OF_WEEK|PICKUP_HOUR|NUM_TRIPS|
+-----------+-----------+---------+
|1          |0          |6370098  |
|1          |1          |5618907  |
|1          |2          |4393571  |
|1          |3          |3385406  |
|1          |4          |2177778  |
|1          |5          |1048054  |
|1          |6          |1156485  |
|1          |7          |1605340  |
|1          |8          |2397859  |
|1          |9          |3571018  |
|1          |10         |4826143  |
|1          |11         |5571376  |
|1          |12         |6101309  |
|1          |13         |6201417  |
|1          |14         |6308764  |
|1          |15         |6175787  |
|1          |16         |6078173  |
|1          |17         |6242361  |
|1          |18         |6315267  |
|1          |19         |5691372  |
+-----------+-----------+---------+
only showing top 20 rows



In [19]:
query_obt("""
    SELECT day_of_week, pickup_hour, COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY day_of_week, pickup_hour
    ORDER BY day_of_week, pickup_hour
""").show(truncate=False)

+-----------+-----------+---------+
|DAY_OF_WEEK|PICKUP_HOUR|NUM_TRIPS|
+-----------+-----------+---------+
|1          |0          |6370098  |
|1          |1          |5618907  |
|1          |2          |4393571  |
|1          |3          |3385406  |
|1          |4          |2177778  |
|1          |5          |1048054  |
|1          |6          |1156485  |
|1          |7          |1605340  |
|1          |8          |2397859  |
|1          |9          |3571018  |
|1          |10         |4826143  |
|1          |11         |5571376  |
|1          |12         |6101309  |
|1          |13         |6201417  |
|1          |14         |6308764  |
|1          |15         |6175787  |
|1          |16         |6078173  |
|1          |17         |6242361  |
|1          |18         |6315267  |
|1          |19         |5691372  |
+-----------+-----------+---------+
only showing top 20 rows



## f) p50/p90 de trip_duration_min por borough de pickup

In [ ]:
result = (
    df.groupby("pu_borough")
      .agg(
          F.round(F.percentile_approx("trip_duration_min", 0.5), 2).alias("p50_duration"),
          F.round(F.percentile_approx("trip_duration_min", 0.9), 2).alias("p90_duration")
      ) 
      .withColumn("ratio", F.round(F.col("p50_duration") / F.col("p90_duration"), 2))
      .orderBy(F.col("p50_duration").desc())
)

result.show(truncate=False)

In [20]:
raw = query_obt("""
    SELECT pu_borough,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p50_duration,
           ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p90_duration,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min) / PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) as ratio
    FROM OBT_TRIPS
    GROUP BY pu_borough
""")

(raw.orderBy(F.col("p50_duration").desc())
   .show(truncate=False)
)

+-------------+------------+------------+-----+
|PU_BOROUGH   |P50_DURATION|P90_DURATION|RATIO|
+-------------+------------+------------+-----+
|Queens       |24.78       |54.92       |0.45 |
|Staten Island|19.75       |72.28       |0.27 |
|Brooklyn     |12.43       |31.52       |0.39 |
|Bronx        |12.32       |38.02       |0.32 |
|Manhattan    |10.77       |25.0        |0.43 |
|Unknown      |10.55       |28.13       |0.38 |
|N/A          |0.83        |23.03       |0.04 |
|EWR          |0.27        |1.87        |0.14 |
+-------------+------------+------------+-----+



In [49]:
query_obt("""
    SELECT pu_borough,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p50_duration,
           ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p90_duration,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min) / PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) as ratio
    FROM OBT_TRIPS
    GROUP BY pu_borough
    ORDER BY p50_duration DESC
""").show(truncate=False)

+-------------+------------+------------+-----+
|PU_BOROUGH   |P50_DURATION|P90_DURATION|RATIO|
+-------------+------------+------------+-----+
|Queens       |24.78       |54.92       |0.45 |
|Staten Island|19.75       |72.28       |0.27 |
|Brooklyn     |12.43       |31.52       |0.39 |
|Bronx        |12.32       |38.02       |0.32 |
|Manhattan    |10.77       |25.0        |0.43 |
|Unknown      |10.55       |28.13       |0.38 |
|N/A          |0.83        |23.03       |0.04 |
|EWR          |0.27        |1.87        |0.14 |
+-------------+------------+------------+-----+



## g) avg_speed_mph por franja horaria (6-9, 17-20) y borough

In [ ]:
result = (
    df.withColumn("pickup_period",
           F.when(F.col("pickup_hour").between(6, 9), "morning")
            .when(F.col("pickup_hour").between(17, 20), "evening")
      )
      .filter(F.col("pickup_period").isNotNull())
      .groupby("pu_borough", "pickup_period")
      .agg(F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed"))
      .orderBy("pu_borough", "pickup_period")
)

result.show(truncate=False)

In [21]:
raw = query_obt("""
    SELECT pu_borough AS borough, AVG(avg_speed_mph),
        CASE 
            WHEN pickup_hour BETWEEN 6 AND 9 THEN 'morning'
            WHEN pickup_hour BETWEEN 17 AND 20 THEN 'evening' 
        END AS pickup_period
    FROM OBT_TRIPS
    WHERE (pickup_hour BETWEEN 6 AND 9) 
    OR (pickup_hour BETWEEN 17 AND 20)
    GROUP BY pu_borough, pickup_period
""")

(raw.orderBy("borough", "pickup_period")
   .show(truncate=False)
)

+-------------+--------------------+-------------+
|BOROUGH      |"AVG(AVG_SPEED_MPH)"|PICKUP_PERIOD|
+-------------+--------------------+-------------+
|Bronx        |12.610981433620159  |evening      |
|Bronx        |13.59120657607936   |morning      |
|Brooklyn     |11.23595257366326   |evening      |
|Brooklyn     |13.053898531766496  |morning      |
|EWR          |25.89858516890895   |evening      |
|EWR          |26.092193833558827  |morning      |
|Manhattan    |9.965971165975878   |evening      |
|Manhattan    |11.385524959845378  |morning      |
|N/A          |20.321938609039123  |evening      |
|N/A          |20.891526465008415  |morning      |
|Queens       |18.821560085584583  |evening      |
|Queens       |18.42522069882059   |morning      |
|Staten Island|21.312485707781452  |evening      |
|Staten Island|21.040214573002597  |morning      |
|Unknown      |11.032623326156681  |evening      |
|Unknown      |12.145485835881713  |morning      |
+-------------+----------------

In [22]:
query_obt("""
    SELECT pu_borough AS borough, AVG(avg_speed_mph),
        CASE 
            WHEN pickup_hour BETWEEN 6 AND 9 THEN 'morning'
            WHEN pickup_hour BETWEEN 17 AND 20 THEN 'evening' 
        END AS pickup_period
    FROM OBT_TRIPS
    WHERE (pickup_hour BETWEEN 6 AND 9) 
    OR (pickup_hour BETWEEN 17 AND 20)
    GROUP BY pu_borough, pickup_period
    ORDER BY pu_borough, pickup_period
""").show(truncate=False)

+-------------+--------------------+-------------+
|BOROUGH      |"AVG(AVG_SPEED_MPH)"|PICKUP_PERIOD|
+-------------+--------------------+-------------+
|Bronx        |12.610981433620157  |evening      |
|Bronx        |13.59120657607936   |morning      |
|Brooklyn     |11.235952573663257  |evening      |
|Brooklyn     |13.053898531766494  |morning      |
|EWR          |25.89858516890895   |evening      |
|EWR          |26.092193833558827  |morning      |
|Manhattan    |9.965971165975876   |evening      |
|Manhattan    |11.385524959845378  |morning      |
|N/A          |20.321938609039126  |evening      |
|N/A          |20.891526465008415  |morning      |
|Queens       |18.821560085584583  |evening      |
|Queens       |18.425220698820585  |morning      |
|Staten Island|21.312485707781452  |evening      |
|Staten Island|21.040214573002597  |morning      |
|Unknown      |11.032623326156681  |evening      |
|Unknown      |12.145485835881713  |morning      |
+-------------+----------------

## h) Participacion por payment_type_desc y su relacion con tip_pct

In [ ]:
result = (
    df.groupby("payment_type_desc")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("tip_pct"), 2)
      )
      .withColumn("pct_trips", F.round((F.count("*") * 100)/ F.sum(F.count(*).over()), 2))
      .orderBy(F.col("num_trips").desc())
)

result.show(truncate=False)

In [23]:
raw = query_obt("""
    SELECT payment_type_desc,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_trips,
           ROUND(AVG(tip_pct), 2) as avg_tip_pct
    FROM OBT_TRIPS
    WHERE payment_type_desc IS NOT NULL
    GROUP BY payment_type_desc
""")

(raw.orderBy(F.col("num_trips").desc())
   .show(truncate=False)
)

+-----------------+---------+---------+-----------+
|PAYMENT_TYPE_DESC|NUM_TRIPS|PCT_TRIPS|AVG_TIP_PCT|
+-----------------+---------+---------+-----------+
|Credit card      |581071805|68.91    |15.05      |
|Cash             |256422051|30.41    |0.0        |
|No charge        |3401728  |0.40     |0.02       |
|Dispute          |2279939  |0.27     |0.02       |
|Unknown          |2693     |0.00     |0.11       |
+-----------------+---------+---------+-----------+



In [24]:
query_obt("""
    SELECT payment_type_desc,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_trips,
           ROUND(AVG(tip_pct), 2) as avg_tip_pct
    FROM OBT_TRIPS
    WHERE payment_type_desc IS NOT NULL
    GROUP BY payment_type_desc
    ORDER BY num_trips DESC
""").show(truncate=False)

+-----------------+---------+---------+-----------+
|PAYMENT_TYPE_DESC|NUM_TRIPS|PCT_TRIPS|AVG_TIP_PCT|
+-----------------+---------+---------+-----------+
|Credit card      |581071805|68.91    |15.05      |
|Cash             |256422051|30.41    |0.0        |
|No charge        |3401728  |0.40     |0.02       |
|Dispute          |2279939  |0.27     |0.02       |
|Unknown          |2693     |0.00     |0.11       |
+-----------------+---------+---------+-----------+



## i) Rate codes con mayor trip_distance y total_amount

In [ ]:
result = (
    df.groupby("rate_code_desc")
      .agg(
          F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
          F.round(F.avg("total_amount"), 2).alias("avg_total"),
          F.round(F.avg("trip_distance"), 2).alias("sum_distance"),
          F.round(F.avg("total_amount"), 2).alias("sum_total")
          
      )
      .orderBy(F.col("sum_total").desc())
)

result.show(truncate=False)

In [25]:
raw = query_obt("""
    SELECT rate_code_desc,
           ROUND(SUM(trip_distance), 2) as sum_distance,
           ROUND(AVG(trip_distance), 2) as avg_distance,
           ROUND(SUM(total_amount), 2) as sum_total,
           ROUND(AVG(total_amount), 2) as avg_total,
    FROM OBT_TRIPS
    GROUP BY rate_code_desc
""")

(raw.orderBy(F.col("sum_total").desc())
   .show(truncate=False)
)

+---------------------+--------------+------------+-----------------+---------+
|RATE_CODE_DESC       |SUM_DISTANCE  |AVG_DISTANCE|SUM_TOTAL        |AVG_TOTAL|
+---------------------+--------------+------------+-----------------+---------+
|Standard rate        |2.1319058986E9|2.62        |1.362839038755E10|16.74    |
|JFK                  |3.4090246605E8|17.18       |1.42604566298E9  |71.85    |
|Negotiated fare      |2.759118287E7 |5.36        |2.9846262604E8   |58.0     |
|Newark               |2.821656051E7 |16.14       |1.6567207939E8   |94.77    |
|Nassau or Westchester|1.427772857E7 |19.47       |7.538154911E7    |102.77   |
|Null/unknown         |1.274705703E7 |7.69        |6.298526359E7    |38.0     |
|Group ride           |3559.07       |0.58        |178623.52        |28.99    |
+---------------------+--------------+------------+-----------------+---------+



In [26]:
query_obt("""
    SELECT rate_code_desc,
           ROUND(SUM(trip_distance), 2) as sum_distance,
           ROUND(AVG(trip_distance), 2) as avg_distance,
           ROUND(SUM(total_amount), 2) as sum_total,
           ROUND(AVG(total_amount), 2) as avg_total,
    FROM OBT_TRIPS
    GROUP BY rate_code_desc
    ORDER BY sum_total DESC
""").show(truncate=False)

+---------------------+--------------+------------+-----------------+---------+
|RATE_CODE_DESC       |SUM_DISTANCE  |AVG_DISTANCE|SUM_TOTAL        |AVG_TOTAL|
+---------------------+--------------+------------+-----------------+---------+
|Standard rate        |2.1319058986E9|2.62        |1.362839038755E10|16.74    |
|JFK                  |3.4090246605E8|17.18       |1.42604566298E9  |71.85    |
|Negotiated fare      |2.759118287E7 |5.36        |2.9846262604E8   |58.0     |
|Newark               |2.821656051E7 |16.14       |1.6567207939E8   |94.77    |
|Nassau or Westchester|1.427772857E7 |19.47       |7.538154911E7    |102.77   |
|Null/unknown         |1.274705703E7 |7.69        |6.298526359E7    |38.0     |
|Group ride           |3559.07       |0.58        |178623.52        |28.99    |
+---------------------+--------------+------------+-----------------+---------+



## j) Mix yellow vs green por mes y borough

In [ ]:
result = (
    df.groupby("pu_borough", "source_year", "source_month", "service_type")
      .agg(
          F.count("*").alias("num_trips")
      )
      .orderBy("sum_total", "source_month", "service_type")
)

result.show(truncate=False)

In [27]:
raw = query_obt("""
    SELECT pu_borough, source_year, source_month, service_type,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month, service_type
""")

(raw.orderBy("source_year", "source_month", "service_type")
   .show(truncate=False)
)

+-------------+-----------+------------+------------+---------+
|PU_BOROUGH   |SOURCE_YEAR|SOURCE_MONTH|SERVICE_TYPE|NUM_TRIPS|
+-------------+-----------+------------+------------+---------+
|Brooklyn     |2015       |1           |green       |568592   |
|Staten Island|2015       |1           |green       |220      |
|Unknown      |2015       |1           |green       |2402     |
|Bronx        |2015       |1           |green       |90934    |
|N/A          |2015       |1           |green       |780      |
|Queens       |2015       |1           |green       |411537   |
|Manhattan    |2015       |1           |green       |429158   |
|EWR          |2015       |1           |green       |27       |
|Brooklyn     |2015       |1           |yellow      |229492   |
|EWR          |2015       |1           |yellow      |593      |
|Queens       |2015       |1           |yellow      |635218   |
|Manhattan    |2015       |1           |yellow      |11606306 |
|Staten Island|2015       |1           |

In [28]:
query_obt("""
    SELECT pu_borough, source_year, source_month, service_type,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month, service_type
    ORDER BY source_year, source_month, service_type
""").show(truncate=False)

+-------------+-----------+------------+------------+---------+
|PU_BOROUGH   |SOURCE_YEAR|SOURCE_MONTH|SERVICE_TYPE|NUM_TRIPS|
+-------------+-----------+------------+------------+---------+
|Staten Island|2015       |1           |green       |220      |
|Queens       |2015       |1           |green       |411537   |
|N/A          |2015       |1           |green       |780      |
|Brooklyn     |2015       |1           |green       |568592   |
|Manhattan    |2015       |1           |green       |429158   |
|Bronx        |2015       |1           |green       |90934    |
|Unknown      |2015       |1           |green       |2402     |
|EWR          |2015       |1           |green       |27       |
|EWR          |2015       |1           |yellow      |593      |
|Brooklyn     |2015       |1           |yellow      |229492   |
|Unknown      |2015       |1           |yellow      |240210   |
|N/A          |2015       |1           |yellow      |7312     |
|Bronx        |2015       |1           |

## k) Top 20 flujos PU→DO por volumen y ticket promedio

In [ ]:
result = (
    df.groupby("pu_zone", "do_zone")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("total_amount"), 2).alias("avg_total")
      )
      .orderBy(F.col("num_trips").desc())
      .limit(20)
)

result.show(truncate=False)

In [29]:
raw = query_obt("""
    SELECT pu_zone, do_zone,
           COUNT(*) as num_trips,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY pu_zone, do_zone
""")

(raw.orderBy(F.col("num_trips").desc())
    .limit(20)
    .show(truncate=False)
)

+----------------------------+----------------------------+---------+---------+
|PU_ZONE                     |DO_ZONE                     |NUM_TRIPS|AVG_TOTAL|
+----------------------------+----------------------------+---------+---------+
|N/A                         |N/A                         |7642897  |17.69    |
|Upper East Side South       |Upper East Side North       |4551125  |10.47    |
|Upper East Side North       |Upper East Side South       |3879650  |11.36    |
|Upper East Side North       |Upper East Side North       |3580180  |8.74     |
|Upper East Side South       |Upper East Side South       |3435853  |9.35     |
|Upper West Side South       |Upper West Side North       |2016975  |8.98     |
|Upper West Side South       |Lincoln Square East         |1999720  |9.52     |
|Upper East Side South       |Midtown Center              |1945105  |12.29    |
|Upper East Side South       |Midtown East                |1937613  |10.85    |
|Lincoln Square East         |Upper West

In [30]:
query_obt("""
    SELECT pu_zone, do_zone,
           COUNT(*) as num_trips,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY pu_zone, do_zone
    ORDER BY num_trips DESC
    LIMIT 20
""").show(truncate=False)

+----------------------------+----------------------------+---------+---------+
|PU_ZONE                     |DO_ZONE                     |NUM_TRIPS|AVG_TOTAL|
+----------------------------+----------------------------+---------+---------+
|N/A                         |N/A                         |7642897  |17.69    |
|Upper East Side South       |Upper East Side North       |4551125  |10.47    |
|Upper East Side North       |Upper East Side South       |3879650  |11.36    |
|Upper East Side North       |Upper East Side North       |3580180  |8.74     |
|Upper East Side South       |Upper East Side South       |3435853  |9.35     |
|Upper West Side South       |Upper West Side North       |2016975  |8.98     |
|Upper West Side South       |Lincoln Square East         |1999720  |9.52     |
|Upper East Side South       |Midtown Center              |1945105  |12.29    |
|Upper East Side South       |Midtown East                |1937613  |10.85    |
|Lincoln Square East         |Upper West

## l) Distribucion de passenger_count y efecto en total_amount

In [ ]:
w = Window.partitionBy() 

(
    df.groupby("passenger_count")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("total_amount"), 2).alias("avg_total")
      )
      .withColumn("pct",
          F.round(F.col("num_trips") * 100 / F.sum("num_trips").over(w), 2)
      )
      .orderBy("passenger_count")
      .show(truncate=False)
)

In [31]:
raw = query_obt("""
    SELECT passenger_count,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY passenger_count
""")

(raw.orderBy("passenger_count")
    .show(truncate=False)
)

+---------------+---------+-----+---------+
|PASSENGER_COUNT|NUM_TRIPS|PCT  |AVG_TOTAL|
+---------------+---------+-----+---------+
|0              |5920239  |0.70 |19.83    |
|1              |615638318|73.01|18.37    |
|2              |118722126|14.08|19.82    |
|3              |32843018 |3.90 |19.24    |
|4              |15943879 |1.89 |20.37    |
|5              |33507704 |3.97 |17.08    |
|6              |20593246 |2.44 |16.9     |
|7              |3778     |0.00 |47.33    |
|8              |3901     |0.00 |49.97    |
|9              |2007     |0.00 |62.46    |
+---------------+---------+-----+---------+



In [32]:
query_obt("""
    SELECT passenger_count,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY passenger_count
    ORDER BY passenger_count
""").show(truncate=False)

+---------------+---------+-----+---------+
|PASSENGER_COUNT|NUM_TRIPS|PCT  |AVG_TOTAL|
+---------------+---------+-----+---------+
|0              |5920239  |0.70 |19.83    |
|1              |615638318|73.01|18.37    |
|2              |118722126|14.08|19.82    |
|3              |32843018 |3.90 |19.24    |
|4              |15943879 |1.89 |20.37    |
|5              |33507704 |3.97 |17.08    |
|6              |20593246 |2.44 |16.9     |
|7              |3778     |0.00 |47.33    |
|8              |3901     |0.00 |49.97    |
|9              |2007     |0.00 |62.46    |
+---------------+---------+-----+---------+



## m) Impacto de tolls_amount y congestion_surcharge por zona

In [ ]:
result = (
    df.groupby("pu_zone")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("congestion_surcharge"), 2).alias("avg_congestion")
          F.round(F.avg("tolls_amount"), 2).alias("avg_tolls")
      )
      .orderBy(F.col("avg_congestion").desc())
)

result.show(truncate=False)

In [33]:
raw = query_obt("""
    SELECT pu_zone,
           ROUND(AVG(tolls_amount), 2) as avg_tolls,
           ROUND(AVG(congestion_surcharge), 2) as avg_congestion,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
""")

(raw.orderBy(F.col("avg_congestion").desc())
    .show(truncate=False)
)

+-----------------------------+---------+--------------+---------+
|PU_ZONE                      |AVG_TOLLS|AVG_CONGESTION|NUM_TRIPS|
+-----------------------------+---------+--------------+---------+
|West Village                 |0.09     |2.96          |16311666 |
|Greenwich Village South      |0.06     |2.96          |10629558 |
|Upper East Side South        |0.07     |2.96          |32503537 |
|Yorkville East               |0.19     |2.95          |9493392  |
|Greenwich Village North      |0.1      |2.95          |12333198 |
|Little Italy/NoLiTa          |0.05     |2.95          |8286857  |
|West Chelsea/Hudson Yards    |0.16     |2.95          |11760185 |
|East Village                 |0.08     |2.95          |21932679 |
|Lenox Hill West              |0.09     |2.95          |18451557 |
|Upper West Side South        |0.13     |2.95          |20031255 |
|SoHo                         |0.08     |2.95          |6371415  |
|Penn Station/Madison Sq West |0.19     |2.95          |267113

In [34]:
query_obt("""
    SELECT pu_zone,
           ROUND(AVG(tolls_amount), 2) as avg_tolls,
           ROUND(AVG(congestion_surcharge), 2) as avg_congestion,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
    ORDER BY avg_congestion DESC
""").show(truncate=False)

+-----------------------------+---------+--------------+---------+
|PU_ZONE                      |AVG_TOLLS|AVG_CONGESTION|NUM_TRIPS|
+-----------------------------+---------+--------------+---------+
|West Village                 |0.09     |2.96          |16311666 |
|Greenwich Village South      |0.06     |2.96          |10629558 |
|Upper East Side South        |0.07     |2.96          |32503537 |
|Little Italy/NoLiTa          |0.05     |2.95          |8286857  |
|Upper West Side South        |0.13     |2.95          |20031255 |
|SoHo                         |0.08     |2.95          |6371415  |
|West Chelsea/Hudson Yards    |0.16     |2.95          |11760185 |
|Lincoln Square East          |0.11     |2.95          |23124484 |
|Penn Station/Madison Sq West |0.19     |2.95          |26711365 |
|Sutton Place/Turtle Bay North|0.11     |2.95          |14896209 |
|East Village                 |0.08     |2.95          |21932679 |
|Battery Park                 |0.2      |2.95          |323622

## n) Proporcion viajes cortos vs largos por borough y estacionalidad

In [ ]:
result = (
    df.withColumn("season",
           F.when(F.col("month").isin(12, 1, 2), "Winter")
            .when(F.col("month").isin(3, 4, 5), "Spring")
            .when(F.col("month").isin(6, 7, 8), "Summer")
            .otherwise("Fall")
      )
      .groupby("pu_borough", "season")
      .agg(
          F.round(F.count("*"), 2).alias("total_trips"),
          F.round((F.sum(F.when(F.col("trip_distance") <= 2, 1).otherwise(0)) * 100) / F.count("*"), 2).alias("pct_short"),
          F.round((F.sum(F.when(F.col("trip_distance") > 10, 1).otherwise(0)) * 100) /  F.count("*"), 2).alias("pct_long")
      )
      .orderBy("pu_borough", "season")
)

result.show(truncate=False)

In [35]:
raw = query_obt("""
    SELECT pu_borough,
           CASE
               WHEN month IN (12, 1, 2) THEN 'Winter'
               WHEN month IN (3, 4, 5) THEN 'Spring'
               WHEN month IN (6, 7, 8) THEN 'Summer'
               ELSE 'Fall'
           END as season,
           ROUND(SUM(CASE WHEN trip_distance <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_short,
           ROUND(SUM(CASE WHEN trip_distance > 10 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_long,
           COUNT(*) as total_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, season
""")

(raw.orderBy("pu_borough", "season")
    .show(truncate=False)
)

+----------+------+---------+--------+-----------+
|PU_BOROUGH|SEASON|PCT_SHORT|PCT_LONG|TOTAL_TRIPS|
+----------+------+---------+--------+-----------+
|Bronx     |Fall  |44.00    |10.99   |958693     |
|Bronx     |Spring|47.03    |8.11    |1296406    |
|Bronx     |Summer|45.78    |9.10    |1065628    |
|Bronx     |Winter|46.95    |8.88    |1124345    |
|Brooklyn  |Fall  |44.11    |5.15    |7455204    |
|Brooklyn  |Spring|44.36    |4.61    |9260259    |
|Brooklyn  |Summer|43.30    |4.87    |7792099    |
|Brooklyn  |Winter|46.44    |4.57    |8363666    |
|EWR       |Fall  |95.88    |3.30    |18766      |
|EWR       |Spring|95.56    |3.38    |16671      |
|EWR       |Summer|94.98    |3.95    |17098      |
|EWR       |Winter|95.81    |3.23    |15975      |
|Manhattan |Fall  |62.23    |2.60    |177534151  |
|Manhattan |Spring|62.08    |2.58    |191697657  |
|Manhattan |Summer|61.43    |2.63    |168538871  |
|Manhattan |Winter|63.65    |2.34    |186456650  |
|N/A       |Fall  |79.84    |8.

In [36]:
query_obt("""
    SELECT pu_borough,
           CASE
               WHEN month IN (12, 1, 2) THEN 'Winter'
               WHEN month IN (3, 4, 5) THEN 'Spring'
               WHEN month IN (6, 7, 8) THEN 'Summer'
               ELSE 'Fall'
           END as season,
           ROUND(SUM(CASE WHEN trip_distance <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_short,
           ROUND(SUM(CASE WHEN trip_distance > 10 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_long,
           COUNT(*) as total_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, season
    ORDER BY pu_borough, season
""").show(truncate=False)

+----------+------+---------+--------+-----------+
|PU_BOROUGH|SEASON|PCT_SHORT|PCT_LONG|TOTAL_TRIPS|
+----------+------+---------+--------+-----------+
|Bronx     |Fall  |44.00    |10.99   |958693     |
|Bronx     |Spring|47.03    |8.11    |1296406    |
|Bronx     |Summer|45.78    |9.10    |1065628    |
|Bronx     |Winter|46.95    |8.88    |1124345    |
|Brooklyn  |Fall  |44.11    |5.15    |7455204    |
|Brooklyn  |Spring|44.36    |4.61    |9260259    |
|Brooklyn  |Summer|43.30    |4.87    |7792099    |
|Brooklyn  |Winter|46.44    |4.57    |8363666    |
|EWR       |Fall  |95.88    |3.30    |18766      |
|EWR       |Spring|95.56    |3.38    |16671      |
|EWR       |Summer|94.98    |3.95    |17098      |
|EWR       |Winter|95.81    |3.23    |15975      |
|Manhattan |Fall  |62.23    |2.60    |177534151  |
|Manhattan |Spring|62.08    |2.58    |191697657  |
|Manhattan |Summer|61.43    |2.63    |168538871  |
|Manhattan |Winter|63.65    |2.34    |186456650  |
|N/A       |Fall  |79.84    |8.

## o) Diferencias por vendor en avg_speed_mph y trip_duration_min

In [ ]:
result = (
    df.groupby("vendor_name")
      .agg(
          F.round(F.count("*"), 2).alias("num_trips"),
          F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed"),
          F.round(F.avg("trip_duration_min"), 2).alias("avg_duration")
      )
      .orderBy(F.col("avg_speed").desc())
)

result.show(truncate=False)

In [37]:
raw = query_obt("""
    SELECT vendor_name,
           ROUND(AVG(avg_speed_mph), 2) as avg_speed,
           ROUND(AVG(trip_duration_min), 2) as avg_duration,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY vendor_name
""")

(raw.orderBy(F.col("avg_speed").desc())
    .show(truncate=False)
)

+---------------------------------+---------+------------+---------+
|VENDOR_NAME                      |AVG_SPEED|AVG_DURATION|NUM_TRIPS|
+---------------------------------+---------+------------+---------+
|Curb Mobility, LLC               |11.76    |18.56       |520139763|
|Creative Mobile Technologies, LLC|11.32    |14.38       |321733276|
|Unknown                          |10.64    |14.17       |769046   |
|Helix                            |NULL     |0.0         |536131   |
+---------------------------------+---------+------------+---------+



In [38]:
query_obt("""
    SELECT vendor_name,
           ROUND(AVG(avg_speed_mph), 2) as avg_speed,
           ROUND(AVG(trip_duration_min), 2) as avg_duration,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY vendor_name
    ORDER BY avg_speed DESC
""").show(truncate=False)

+---------------------------------+---------+------------+---------+
|VENDOR_NAME                      |AVG_SPEED|AVG_DURATION|NUM_TRIPS|
+---------------------------------+---------+------------+---------+
|Helix                            |NULL     |0.0         |536131   |
|Curb Mobility, LLC               |11.76    |18.56       |520139763|
|Creative Mobile Technologies, LLC|11.32    |14.38       |321733276|
|Unknown                          |10.64    |14.17       |769046   |
+---------------------------------+---------+------------+---------+



## p) Relacion metodo de pago y tip_amount por hora

In [ ]:
result = (
    df.groupby("payment_type_desc", "pickup_hour")
      .agg(
          F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
          F.round(F.count("*"), 2).alias("num_trips")
      )
      .orderBy("payment_type_desc", "pickup_hour")
)

result.show(truncate=False)

In [39]:
raw = query_obt("""
    SELECT payment_type_desc, pickup_hour,
           ROUND(AVG(tip_amount), 2) as avg_tip,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY payment_type_desc, pickup_hour
""")

(raw.orderBy(F.col("avg_tip").desc())
    .show(truncate=False)
)

+-----------------+-----------+-------+---------+
|PAYMENT_TYPE_DESC|PICKUP_HOUR|AVG_TIP|NUM_TRIPS|
+-----------------+-----------+-------+---------+
|Unknown          |12         |7.26   |138      |
|Credit card      |5          |3.8    |4732569  |
|Credit card      |16         |3.43   |29264985 |
|Credit card      |4          |3.33   |4381142  |
|Credit card      |15         |3.26   |30542545 |
|Credit card      |14         |3.25   |30466463 |
|Credit card      |17         |3.23   |33874268 |
|Credit card      |13         |3.15   |28842023 |
|Credit card      |23         |3.14   |26051880 |
|Credit card      |22         |3.11   |31828282 |
|Credit card      |21         |3.06   |33896638 |
|Credit card      |19         |3.06   |37279778 |
|Credit card      |12         |3.04   |28670297 |
|Credit card      |0          |3.04   |19115866 |
|Credit card      |18         |3.03   |38486047 |
|Credit card      |11         |3.0    |27067551 |
|Credit card      |6          |2.98   |11333880 |


In [40]:
query_obt("""
    SELECT payment_type_desc, pickup_hour,
           ROUND(AVG(tip_amount), 2) as avg_tip,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY payment_type_desc, pickup_hour
    ORDER BY payment_type_desc, pickup_hour
""").show(truncate=False)

+-----------------+-----------+-------+---------+
|PAYMENT_TYPE_DESC|PICKUP_HOUR|AVG_TIP|NUM_TRIPS|
+-----------------+-----------+-------+---------+
|Cash             |0          |0.0    |8248283  |
|Cash             |1          |0.0    |6054225  |
|Cash             |2          |0.0    |4464745  |
|Cash             |3          |0.0    |3484263  |
|Cash             |4          |0.0    |3189885  |
|Cash             |5          |0.0    |2871353  |
|Cash             |6          |0.0    |5300364  |
|Cash             |7          |0.0    |7890084  |
|Cash             |8          |0.0    |9467688  |
|Cash             |9          |0.0    |10916106 |
|Cash             |10         |0.0    |12468684 |
|Cash             |11         |0.0    |13516609 |
|Cash             |12         |0.0    |14337362 |
|Cash             |13         |0.0    |14652956 |
|Cash             |14         |0.0    |15472957 |
|Cash             |15         |0.0    |15371649 |
|Cash             |16         |0.0    |14151641 |


## q) Zonas con percentil 99 de duracion/distancia fuera de rango

In [ ]:
result = (
    df.groupby("pu_zone")
      .agg(
          F.round(F.percentile_approx("trip_duration_min", 0.99), 2).alias("p99_duration"),
          F.round(F.percentile_approx("trip_distance", 0.99), 2).alias("p99_distance"),
          F.round(F.count("*"), 2).alias("num_trips")
      ) 
      .orderBy(F.col("p99_duration").desc())
)

result.show(truncate=False)

In [41]:
raw = query_obt("""
    SELECT pu_zone,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p99_duration,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance), 2) AS p99_distance,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
""")

(raw.orderBy(F.col("p99_duration").desc())
    .show(truncate=False)
)

+---------------------------------+------------+------------+---------+
|PU_ZONE                          |P99_DURATION|P99_DISTANCE|NUM_TRIPS|
+---------------------------------+------------+------------+---------+
|Bronx Park                       |323.3       |21.6        |31161    |
|Coney Island                     |279.51      |29.04       |162724   |
|West Brighton                    |208.03      |41.47       |1422     |
|Rossville/Woodrow                |194.62      |46.24       |321      |
|Mariners Harbor                  |190.72      |34.78       |4511     |
|Arden Heights                    |188.09      |39.93       |2082     |
|Hammels/Arverne                  |163.96      |32.6        |25184    |
|Eltingville/Annadale/Prince's Bay|152.74      |43.82       |397      |
|Far Rockaway                     |152.13      |31.8        |23129    |
|Rockaway Park                    |147.28      |31.5        |6431     |
|Brighton Beach                   |143.85      |30.07       |499

In [48]:
query_obt("""
    SELECT pu_zone,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p99_duration,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance), 2) AS p99_distance,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
    ORDER BY p99_duration DESC
""").show(truncate=False)

+---------------------------------+------------+------------+---------+
|PU_ZONE                          |P99_DURATION|P99_DISTANCE|NUM_TRIPS|
+---------------------------------+------------+------------+---------+
|Bronx Park                       |323.3       |21.6        |31161    |
|Coney Island                     |279.51      |29.04       |162724   |
|West Brighton                    |208.03      |41.47       |1422     |
|Rossville/Woodrow                |194.62      |46.24       |321      |
|Mariners Harbor                  |190.72      |34.78       |4511     |
|Arden Heights                    |188.09      |39.93       |2082     |
|Hammels/Arverne                  |163.96      |32.6        |25184    |
|Eltingville/Annadale/Prince's Bay|152.74      |43.82       |397      |
|Far Rockaway                     |152.13      |31.8        |23129    |
|Rockaway Park                    |147.28      |31.5        |6431     |
|Brighton Beach                   |143.85      |30.07       |499

## r) Yield por milla (total_amount/trip_distance) por borough y hora

In [ ]:
result = (
    df.withColumn("yield_per_mile",
           F.when(F.col("trip_distance") > 0,
                  F.col("total_amount") / F.col("trip_distance"))
                 )
      .groupby("pu_borough", "pickup_hour")
      .agg(
          F.round(F.count("*"), 2).alias("num_trips"),
          F.round(F.avg("yield_per_mile"), 2).alias("avg_yield_per_mile")
      ) 
      .orderBy("pu_borough", "pickup_hour")
)

result.show(truncate=False)

In [42]:
raw = query_obt("""
    SELECT pu_borough, pickup_hour,
           ROUND(AVG(total_amount / NULLIF(trip_distance, 0)), 2) as yield_per_mile,
           COUNT(*) as trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, pickup_hour
""")

(raw.orderBy("pu_borough", "pickup_hour")
    .show(truncate=False)
)

+----------+-----------+--------------+------+
|PU_BOROUGH|PICKUP_HOUR|YIELD_PER_MILE|TRIPS |
+----------+-----------+--------------+------+
|Bronx     |0          |14.77         |128862|
|Bronx     |1          |17.06         |97346 |
|Bronx     |2          |18.53         |70186 |
|Bronx     |3          |17.18         |57575 |
|Bronx     |4          |15.19         |65661 |
|Bronx     |5          |13.85         |77611 |
|Bronx     |6          |12.68         |127447|
|Bronx     |7          |12.18         |212447|
|Bronx     |8          |12.51         |273995|
|Bronx     |9          |12.5          |254669|
|Bronx     |10         |12.23         |226628|
|Bronx     |11         |12.18         |217939|
|Bronx     |12         |11.51         |223622|
|Bronx     |13         |12.58         |224973|
|Bronx     |14         |12.52         |247934|
|Bronx     |15         |12.8          |262416|
|Bronx     |16         |13.53         |263304|
|Bronx     |17         |13.7          |271784|
|Bronx     |1

In [43]:
query_obt("""
    SELECT pu_borough, pickup_hour,
           ROUND(AVG(total_amount / NULLIF(trip_distance, 0)), 2) as yield_per_mile,
           COUNT(*) as trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, pickup_hour
    ORDER BY pu_borough, pickup_hour
""").show(truncate=False)

+----------+-----------+--------------+------+
|PU_BOROUGH|PICKUP_HOUR|YIELD_PER_MILE|TRIPS |
+----------+-----------+--------------+------+
|Bronx     |0          |14.77         |128862|
|Bronx     |1          |17.06         |97346 |
|Bronx     |2          |18.53         |70186 |
|Bronx     |3          |17.18         |57575 |
|Bronx     |4          |15.19         |65661 |
|Bronx     |5          |13.85         |77611 |
|Bronx     |6          |12.68         |127447|
|Bronx     |7          |12.18         |212447|
|Bronx     |8          |12.51         |273995|
|Bronx     |9          |12.5          |254669|
|Bronx     |10         |12.23         |226628|
|Bronx     |11         |12.18         |217939|
|Bronx     |12         |11.51         |223622|
|Bronx     |13         |12.58         |224973|
|Bronx     |14         |12.52         |247934|
|Bronx     |15         |12.8          |262416|
|Bronx     |16         |13.53         |263304|
|Bronx     |17         |13.7          |271784|
|Bronx     |1

## s) Cambios YoY en volumen y ticket promedio por service_type

In [39]:
yearly = (
    df.groupby("service_type", "source_year")
      .agg(
          F.count("*").alias("trips"),
          F.round(F.avg("total_amount"), 2).alias("avg_total")
      )
)

a = yearly.alias("a")
b = yearly.alias("b")

result = (
    a.join(
        b,
        (F.col("a.service_type") == F.col("b.service_type")) &
        (F.col("a.source_year") == F.col("b.source_year") + 1),
        "left"
    )
    .select(
        F.col("a.service_type"),
        F.col("a.source_year"),
        F.col("a.trips"),
        F.col("a.avg_total"),
        F.round(
            (F.col("a.trips") - F.col("b.trips")) * 100 /
            F.when(F.col("b.trips") != 0, F.col("b.trips")), 2
        ).alias("volume_change_pct"),
        F.round(
            F.col("a.avg_total") - F.col("b.avg_total"), 2
        ).alias("ticket_change")
    )
    .orderBy("service_type", "source_year")
)

result.show(truncate=False)



In [44]:
raw = query_obt("""
    WITH yearly AS (
        SELECT service_type, source_year,
               COUNT(*) as trips,
               ROUND(AVG(total_amount), 2) as avg_total
        FROM OBT_TRIPS
        GROUP BY service_type, source_year
    )
    SELECT
        a.service_type, a.source_year,
        a.trips, a.avg_total,
        ROUND((a.trips - b.trips) * 100.0 / NULLIF(b.trips, 0), 2) as volume_change_pct,
        ROUND(a.avg_total - b.avg_total, 2) as ticket_change
    FROM yearly a
    LEFT JOIN yearly b ON a.service_type = b.service_type AND a.source_year = b.source_year + 1
    ORDER BY a.service_type, a.source_year
""")

(raw.show(truncate=False)
)

+------------+-----------+---------+---------+-----------------+-------------+
|SERVICE_TYPE|SOURCE_YEAR|TRIPS    |AVG_TOTAL|VOLUME_CHANGE_PCT|TICKET_CHANGE|
+------------+-----------+---------+---------+-----------------+-------------+
|green       |2015       |19167212 |14.88    |NULL             |NULL         |
|green       |2016       |16321186 |14.69    |-14.85           |-0.19        |
|green       |2017       |11692295 |14.29    |-28.36           |-0.4         |
|green       |2018       |8763535  |15.81    |-25.05           |1.52         |
|green       |2019       |5598786  |16.06    |-36.11           |0.25         |
|green       |2020       |1199562  |15.79    |-78.57           |-0.27        |
|green       |2021       |652916   |18.97    |-45.57           |3.18         |
|green       |2022       |745809   |18.39    |14.23            |-0.58        |
|green       |2023       |727485   |23.41    |-2.46            |5.02         |
|green       |2024       |632145   |24.07    |-13.11

In [45]:
query_obt("""
    WITH yearly AS (
        SELECT service_type, source_year,
               COUNT(*) as trips,
               ROUND(AVG(total_amount), 2) as avg_total
        FROM OBT_TRIPS
        GROUP BY service_type, source_year
    )
    SELECT
        a.service_type, a.source_year,
        a.trips, a.avg_total,
        ROUND((a.trips - b.trips) * 100.0 / NULLIF(b.trips, 0), 2) as volume_change_pct,
        ROUND(a.avg_total - b.avg_total, 2) as ticket_change
    FROM yearly a
    LEFT JOIN yearly b ON a.service_type = b.service_type AND a.source_year = b.source_year + 1
    ORDER BY a.service_type, a.source_year
""").show(truncate=False)

+------------+-----------+---------+---------+-----------------+-------------+
|SERVICE_TYPE|SOURCE_YEAR|TRIPS    |AVG_TOTAL|VOLUME_CHANGE_PCT|TICKET_CHANGE|
+------------+-----------+---------+---------+-----------------+-------------+
|green       |2015       |19167212 |14.88    |NULL             |NULL         |
|green       |2016       |16321186 |14.69    |-14.85           |-0.19        |
|green       |2017       |11692295 |14.29    |-28.36           |-0.4         |
|green       |2018       |8763535  |15.81    |-25.05           |1.52         |
|green       |2019       |5598786  |16.06    |-36.11           |0.25         |
|green       |2020       |1199562  |15.79    |-78.57           |-0.27        |
|green       |2021       |652916   |18.97    |-45.57           |3.18         |
|green       |2022       |745809   |18.39    |14.23            |-0.58        |
|green       |2023       |727485   |23.41    |-2.46            |5.02         |
|green       |2024       |632145   |24.07    |-13.11

## t) Dias con alta congestion_surcharge: efecto en total_amount vs dias normales

In [ ]:
daily = (
    df.filter(
        (F.col("congestion_surcharge").isNotNull()) &
        (F.col("pickup_date").isNotNull())
    )
    .groupby("pickup_date")
    .agg(
        F.avg("congestion_surcharge").alias("avg_congestion"),
        F.avg("total_amount").alias("avg_total"),
        F.count("*").alias("num_trips")
    )
)

# 2. Clasificación + agregación final
result = (
    daily.withColumn(
        "congestion_level",
        F.when(F.col("avg_congestion") > 2.0, "Alta congestion")
         .otherwise("Normal")
    )
    .groupby("congestion_level")
    .agg(
        F.count("*").alias("num_days"),
        F.round(F.avg("avg_total"), 2).alias("avg_total_amount"),
        F.round(F.avg("num_trips"), 0).alias("avg_daily_trips")
    )
)

result.show(truncate=False)

In [46]:
raw = query_obt("""
    WITH daily AS (
        SELECT pickup_date,
               AVG(congestion_surcharge) as avg_congestion,
               AVG(total_amount) as avg_total,
               COUNT(*) as num_trips
        FROM OBT_TRIPS
        WHERE congestion_surcharge IS NOT NULL AND pickup_date IS NOT NULL
        GROUP BY pickup_date
    )
    SELECT
        CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END as congestion_level,
        COUNT(*) as num_days,
        ROUND(AVG(avg_total), 2) as avg_total_amount,
        ROUND(AVG(num_trips)) as avg_daily_trips
    FROM daily
    GROUP BY CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END
""")

(raw.show(truncate=False)
)

+----------------+--------+----------------+---------------+
|CONGESTION_LEVEL|NUM_DAYS|AVG_TOTAL_AMOUNT|AVG_DAILY_TRIPS|
+----------------+--------+----------------+---------------+
|Normal          |53      |12.95           |60789          |
|Alta congestion |2549    |23.6            |112106         |
+----------------+--------+----------------+---------------+



In [47]:
query_obt("""
    WITH daily AS (
        SELECT pickup_date,
               AVG(congestion_surcharge) as avg_congestion,
               AVG(total_amount) as avg_total,
               COUNT(*) as num_trips
        FROM OBT_TRIPS
        WHERE congestion_surcharge IS NOT NULL AND pickup_date IS NOT NULL
        GROUP BY pickup_date
    )
    SELECT
        CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END as congestion_level,
        COUNT(*) as num_days,
        ROUND(AVG(avg_total), 2) as avg_total_amount,
        ROUND(AVG(num_trips)) as avg_daily_trips
    FROM daily
    GROUP BY CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END
""").show(truncate=False)

+----------------+--------+----------------+---------------+
|CONGESTION_LEVEL|NUM_DAYS|AVG_TOTAL_AMOUNT|AVG_DAILY_TRIPS|
+----------------+--------+----------------+---------------+
|Normal          |53      |12.95           |60789          |
|Alta congestion |2549    |23.6            |112106         |
+----------------+--------+----------------+---------------+

